# Plugin Config Store

> Persistent storage for per-plugin configuration (with enabled flag)

In [ ]:
#| default_exp core.config_store

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import json
import sqlite3
import time
from contextlib import contextmanager
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, Iterator, Optional, Protocol, runtime_checkable

import logging
_logger = logging.getLogger(__name__)

## PluginConfigRecord

Folded shape that pairs a plugin's config dict with its `enabled` flag. The pairing
(per CR-2's enable/disable design) lives in one record so the substrate can persist
and restore both in a single round-trip.

In [ ]:
#| export
@dataclass
class PluginConfigRecord:
    """Persisted state for a plugin: config dict + enabled flag."""
    config: Dict[str, Any] = field(default_factory=dict)  # Plugin's current config values
    enabled: bool = True  # Whether the substrate should accept jobs for this plugin
    updated_at: float = 0.0  # Unix timestamp of the last write (server clock)

## PluginConfigStore Protocol

Interface that any store implementation must satisfy. Substrate ships
`LocalPluginConfigStore` as the default cross-session, single-user backend; the
future `cjm-workflow-state`-backed `WorkflowPluginConfigStore` (CR-2) implements
the same Protocol so hosts can swap stores without code changes.

`runtime_checkable` so consumers can `isinstance(x, PluginConfigStore)` for
duck-typing, while still relying on static-type-checker enforcement.

In [ ]:
#| export
@runtime_checkable
class PluginConfigStore(Protocol):
    """Protocol for persisting per-plugin `PluginConfigRecord` across sessions."""
    
    def get(self, plugin_name: str) -> Optional[PluginConfigRecord]:
        """Fetch the record for a plugin, or None if no record exists yet."""
        ...
    
    def set(self, plugin_name: str, record: PluginConfigRecord) -> None:
        """Persist a record. Overwrites any prior record for the same plugin.
        
        Implementations stamp `record.updated_at` to the current time during
        the write so callers don't have to manage timestamps.
        """
        ...
    
    def delete(self, plugin_name: str) -> bool:
        """Remove the record for a plugin. Returns True if a record was deleted."""
        ...
    
    def list_all(self) -> Dict[str, PluginConfigRecord]:
        """Return every stored record, keyed by plugin name."""
        ...

## LocalPluginConfigStore

Substrate-shipped default. SQLite at `~/.cjm/plugin_configs.db` (or a caller-provided
path). Cross-session, single-user — suitable for CLI tools and single-user desktop
hosts. Workflow-scoped or multi-user hosts plug a different `PluginConfigStore`
implementation in via dependency injection.

In [ ]:
#| export
_SCHEMA = """
CREATE TABLE IF NOT EXISTS plugin_configs (
    plugin_name TEXT PRIMARY KEY,
    config_json TEXT NOT NULL,
    enabled INTEGER NOT NULL DEFAULT 1,
    updated_at REAL NOT NULL
)
"""


def _default_db_path() -> Path:
    """Default SQLite location: `~/.cjm/plugin_configs.db`."""
    return Path.home() / ".cjm" / "plugin_configs.db"


class LocalPluginConfigStore:
    """SQLite-backed default implementation of `PluginConfigStore`.
    
    The DB is created lazily on first write. Reads against a non-existent DB
    return empty results rather than raising, so hosts can call `.get()` on
    a fresh install without preparing the file first.
    """
    
    def __init__(self, db_path: Optional[Path] = None):
        """Initialize the store. `db_path=None` uses `~/.cjm/plugin_configs.db`."""
        self.db_path = Path(db_path) if db_path is not None else _default_db_path()
    
    @contextmanager
    def _conn(self) -> Iterator[sqlite3.Connection]:
        """Open a connection, creating parent dirs + schema on demand."""
        self.db_path.parent.mkdir(parents=True, exist_ok=True)
        conn = sqlite3.connect(self.db_path)
        try:
            conn.execute(_SCHEMA)
            conn.commit()
            yield conn
        finally:
            conn.close()
    
    def get(
        self,
        plugin_name: str  # Plugin to look up
    ) -> Optional[PluginConfigRecord]:  # Persisted record or None if absent
        """Fetch the record for a plugin."""
        if not self.db_path.exists():
            return None
        with self._conn() as conn:
            row = conn.execute(
                "SELECT config_json, enabled, updated_at FROM plugin_configs WHERE plugin_name = ?",
                (plugin_name,),
            ).fetchone()
        if row is None:
            return None
        config_json, enabled, updated_at = row
        try:
            config = json.loads(config_json) if config_json else {}
        except json.JSONDecodeError as e:
            _logger.warning(
                "Corrupted config row for plugin %s: %s. Returning empty config.",
                plugin_name, e,
            )
            config = {}
        return PluginConfigRecord(
            config=config,
            enabled=bool(enabled),
            updated_at=float(updated_at),
        )
    
    def set(
        self,
        plugin_name: str,  # Plugin to write
        record: PluginConfigRecord  # New record (updated_at overwritten with current time)
    ) -> None:
        """Persist a record. Stamps `updated_at` to the current time."""
        record.updated_at = time.time()
        with self._conn() as conn:
            conn.execute(
                "INSERT OR REPLACE INTO plugin_configs "
                "(plugin_name, config_json, enabled, updated_at) VALUES (?, ?, ?, ?)",
                (
                    plugin_name,
                    json.dumps(record.config),
                    1 if record.enabled else 0,
                    record.updated_at,
                ),
            )
            conn.commit()
    
    def delete(
        self,
        plugin_name: str  # Plugin to remove
    ) -> bool:  # True if a row was deleted
        """Remove the record for a plugin."""
        if not self.db_path.exists():
            return False
        with self._conn() as conn:
            cur = conn.execute(
                "DELETE FROM plugin_configs WHERE plugin_name = ?", (plugin_name,),
            )
            conn.commit()
            return cur.rowcount > 0
    
    def list_all(self) -> Dict[str, PluginConfigRecord]:  # plugin_name -> record
        """Return all stored records keyed by plugin name."""
        if not self.db_path.exists():
            return {}
        with self._conn() as conn:
            rows = conn.execute(
                "SELECT plugin_name, config_json, enabled, updated_at FROM plugin_configs",
            ).fetchall()
        out: Dict[str, PluginConfigRecord] = {}
        for name, config_json, enabled, updated_at in rows:
            try:
                config = json.loads(config_json) if config_json else {}
            except json.JSONDecodeError:
                config = {}
            out[name] = PluginConfigRecord(
                config=config,
                enabled=bool(enabled),
                updated_at=float(updated_at),
            )
        return out

In [ ]:
# SG-22 regression: LocalPluginConfigStore satisfies the PluginConfigStore
# Protocol and round-trips records cleanly across set/get/delete/list_all.
import tempfile
import os

fd, db_path = tempfile.mkstemp(suffix=".db")
os.close(fd)
os.unlink(db_path)  # Start from a fresh non-existent file

try:
    store = LocalPluginConfigStore(Path(db_path))
    
    # Protocol satisfaction (runtime_checkable enables isinstance)
    assert isinstance(store, PluginConfigStore), \
        "LocalPluginConfigStore should satisfy PluginConfigStore Protocol"
    
    # Empty store: missing reads return None and {}
    assert store.get("whisper") is None
    assert store.list_all() == {}
    assert store.delete("whisper") is False
    
    # Round-trip a record
    rec = PluginConfigRecord(config={"model": "large-v3"}, enabled=False)
    store.set("whisper", rec)
    
    out = store.get("whisper")
    assert out is not None
    assert out.config == {"model": "large-v3"}
    assert out.enabled is False
    assert out.updated_at > 0
    
    # Overwrite + list_all
    store.set("whisper", PluginConfigRecord(config={"model": "tiny"}, enabled=True))
    store.set("gemini", PluginConfigRecord(config={"api_key": "x"}, enabled=True))
    all_records = store.list_all()
    assert set(all_records.keys()) == {"whisper", "gemini"}
    assert all_records["whisper"].config == {"model": "tiny"}
    assert all_records["whisper"].enabled is True
    
    # Delete returns True once + False on second call
    assert store.delete("whisper") is True
    assert store.delete("whisper") is False
    assert store.get("whisper") is None
    assert set(store.list_all().keys()) == {"gemini"}
    
    print("SG-22 LocalPluginConfigStore round-trip: PASS")
finally:
    try:
        os.unlink(db_path)
    except OSError:
        pass

SG-22 LocalPluginConfigStore round-trip: PASS


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()